In [1]:
import nltk

nltk.download("gutenberg")
from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to C:\Users\Muhammad
[nltk_data]     Umer\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [2]:
data = gutenberg.raw("shakespeare-hamlet.txt")
with open("shakespeare-hamlet.txt", "w", encoding="utf-8") as f:
    f.write(data)

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [4]:
with open("shakespeare-hamlet.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

In [5]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

In [6]:
# tokenizer.word_index

In [7]:
# Create input sequences using list of tokens
input_sequences = []
for line in text.split("\n"):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        n_gram_sequence = tokens[: i + 1]
        input_sequences.append(n_gram_sequence)

In [8]:
# Pad sequences to ensure uniform input size
max_sequence_len = max([len(x) for x in input_sequences])
max_sequence_len

14

In [9]:
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
)

In [10]:
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]],
      shape=(25732, 14), dtype=int32)

In [11]:
# create predictors and label
# X = all the rows, all the columns except the last column
# y = all the rows, only the last column
X, y = input_sequences[:, :-1], input_sequences[:, -1]

In [12]:
# create one hot encoding of the y variable
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(25732, 4818))

## Training the LSTM


In [19]:
OUT_DIM = 100

model = tf.keras.Sequential(
    [
        tf.keras.layers.Embedding(
            input_dim=total_words,
            output_dim=OUT_DIM,
            input_length=max_sequence_len,
        ),
        tf.keras.layers.LSTM(150, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(100),
        tf.keras.layers.Dense(total_words, activation="softmax"),
    ]
)

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

history = model.fit(
    X,
    y,
    epochs=50,
    verbose=1,
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.0343 - loss: 6.8613
Epoch 2/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.0422 - loss: 6.4525
Epoch 3/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.0511 - loss: 6.2887
Epoch 4/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0559 - loss: 6.1313
Epoch 5/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.0657 - loss: 5.9621
Epoch 6/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0743 - loss: 5.8060
Epoch 7/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.0816 - loss: 5.6707
Epoch 8/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.0888 - loss: 5.5427
Epoch 9/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.0940 - loss: 5.4180
Epoch 10/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.1007 - loss: 5.2903
Epoch 11/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.1061 - loss: 5.1639
Epoch 12/50
805/805 ━━━━━━━━━━━━━━━━━━━━ 10s

## _To use GRU, just change the LSTM to GRU_


In [ ]:
model.save("./next-word-predictor.keras")

In [33]:
import pickle

with open("./next-word-predictor-tokenizer.pickle", "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [34]:
model = tf.keras.models.load_model("./next-word-predictor.keras")

with open("./next-word-predictor-tokenizer.pickle", "rb") as handle:
    tokenizer = pickle.load(handle)

d:\Workspace\ai-ml-dl\ml-ds-krish-naik\24-next-word-predictor\venv\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop_1', because it has 11 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [31]:
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text.lower()])[0]

    # Keep only the last (max_sequence_len - 1) tokens, since that's
    # what the model was trained on as input length
    token_list = token_list[-(max_sequence_len - 1) :]

    token_list = pad_sequences([token_list], maxlen=max_sequence_len, padding="pre")

    predicted_probs = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted_probs, axis=1)

    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word

    return None


# Example usage
seed_text = "Ham. In the secret parts"
next_word = predict_next_word(model, tokenizer, seed_text, max_sequence_len)
print(f"{seed_text} {next_word}")

Ham. In the secret parts of


In [32]:
def generate_text(model, tokenizer, seed_text, max_sequence_len, num_words=10):
    for _ in range(num_words):
        next_word = predict_next_word(model, tokenizer, seed_text, max_sequence_len)
        if next_word is None:
            break
        seed_text += " " + next_word
        print(seed_text)
    return seed_text


final_text = generate_text(
    model, tokenizer, "Ham. In the secret parts", max_sequence_len, num_words=10
)

Ham. In the secret parts of
Ham. In the secret parts of fortune
Ham. In the secret parts of fortune oh
Ham. In the secret parts of fortune oh this
Ham. In the secret parts of fortune oh this thing
Ham. In the secret parts of fortune oh this thing how
Ham. In the secret parts of fortune oh this thing how do'st
Ham. In the secret parts of fortune oh this thing how do'st thou
Ham. In the secret parts of fortune oh this thing how do'st thou beg
Ham. In the secret parts of fortune oh this thing how do'st thou beg me
